# GridWorld Q-Learning Assignment

This notebook trains a Q-learning agent on the 5x5 GridWorld defined in `gridworld.py` and inspects the learned policy.

## Learning objectives
- Understand how the exploration parameter `epsilon` shapes learning.
- Observe how episodic rewards evolve over training.
- Translate the learned Q-values into an interpretable policy.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

# Minimal GridWorld environment used for the assignment
class GridWorld:
    def __init__(self):
        self.grid_size = (5, 5)
        self.start_state = (0, 0)
        self.goal_state = (4, 4)
        self.obstacles = [(1, 1), (2, 2), (3, 1), (3, 3), (1, 3)]
        self.actions = {'UP': 0, 'DOWN': 1, 'LEFT': 2, 'RIGHT': 3}
        self.num_actions = len(self.actions)
        self.num_states = self.grid_size[0] * self.grid_size[1]

    def state_to_coords(self, state):
        return (state // self.grid_size[0], state % self.grid_size[1])

    def coords_to_state(self, coords):
        return coords[0] * self.grid_size[0] + coords[1]

    def is_valid_state(self, state):
        row, col = self.state_to_coords(state)
        return 0 <= row < self.grid_size[0] and 0 <= col < self.grid_size[1]

    def is_obstacle(self, state):
        return self.state_to_coords(state) in self.obstacles

    def is_goal(self, state):
        return self.state_to_coords(state) == self.goal_state

    def get_reward(self, state, next_state):
        if self.is_goal(next_state):
            return 100
        if self.is_obstacle(next_state):
            return -10
        return -1

    def step(self, state, action):
        row, col = self.state_to_coords(state)

        if action == self.actions['UP']:
            next_row, next_col = row - 1, col
        elif action == self.actions['DOWN']:
            next_row, next_col = row + 1, col
        elif action == self.actions['LEFT']:
            next_row, next_col = row, col - 1
        elif action == self.actions['RIGHT']:
            next_row, next_col = row, col + 1
        else:
            raise ValueError("Invalid action")

        next_coords = (next_row, next_col)
        next_state = self.coords_to_state(next_coords)

        if not self.is_valid_state(next_state) or self.is_obstacle(next_state):
            next_state = state
            reward = self.get_reward(state, next_state)
        else:
            reward = self.get_reward(state, next_state)

        done = self.is_goal(next_state)
        return next_state, reward, done

    def reset(self):
        return self.coords_to_state(self.start_state)


class QLearningAgent:
    def __init__(self, env: GridWorld, alpha=0.1, gamma=0.99, epsilon=0.1):
        self.env = env
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.q_table = np.zeros((env.num_states, env.num_actions))

    def choose_action(self, state):
        if random.uniform(0, 1) < self.epsilon:
            return random.choice(range(self.env.num_actions))
        return np.argmax(self.q_table[state, :])

    def update_q_table(self, state, action, reward, next_state):
        old_value = self.q_table[state, action]
        next_max = np.max(self.q_table[next_state, :])
        self.q_table[state, action] = old_value + self.alpha * (reward + self.gamma * next_max - old_value)

    def train(self, num_episodes):
        rewards_per_episode = []

        for episode in range(num_episodes):
            state = self.env.reset()
            done = False
            total_reward = 0

            while not done:
                action = self.choose_action(state)
                next_state, reward, done = self.env.step(state, action)
                self.update_q_table(state, action, reward, next_state)
                state = next_state
                total_reward += reward

            rewards_per_episode.append(total_reward)
            if episode % 500 == 0:
                print(f"Episode {episode}: Total Reward = {total_reward}")

        print("Training finished.")
        return rewards_per_episode

    def get_optimal_policy(self):
        return np.argmax(self.q_table, axis=1)


# Set seeds so runs are reproducible when tuning hyperparameters
np.random.seed(123)
random.seed(123)

env = GridWorld()
agent = QLearningAgent(env, alpha=0.2, gamma=0.95, epsilon=0.2)
episodes = 3000

In [ ]:
rewards = agent.train(episodes)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rewards, color="steelblue")
ax.set_xlabel("Episode")
ax.set_ylabel("Total Reward")
ax.set_title("GridWorld Q-Learning Training Reward")
ax.grid(True)
plt.show()

In [ ]:
action_symbols = {0: '^', 1: 'v', 2: '<', 3: '>'}
policy = agent.get_optimal_policy()
policy_grid = np.full(env.grid_size, '.', dtype=str)

for obstacle in env.obstacles:
    policy_grid[obstacle] = 'X'

for state in range(env.num_states):
    row, col = env.state_to_coords(state)
    if env.is_goal(state):
        policy_grid[row, col] = 'G'
    elif (row, col) == env.start_state:
        policy_grid[row, col] = 'S'
    elif not env.is_obstacle(state):
        policy_grid[row, col] = action_symbols[policy[state]]

print("Greedy policy (arrows show best action):")
for r in range(env.grid_size[0]):
    print(' '.join(policy_grid[r]))

print("\nSample greedy rollout:")
state = env.reset()
trajectory = [env.state_to_coords(state)]
done = False

while not done:
    action = np.argmax(agent.q_table[state])
    state, reward, done = env.step(state, action)
    trajectory.append(env.state_to_coords(state))

print(trajectory)

print("\nQ-values by state:")
for state in range(env.num_states):
    coords = env.state_to_coords(state)
    print(f"{coords}: {agent.q_table[state]}")